In [1]:
import pandas as pd
df = pd.read_csv("https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv")
df.head()
df["Class"].value_counts(normalize=True)

,proportion
Class,
0,0.998273
1,0.001727


In [2]:
#Preparar
import numpy as np
from sklearn.model_selection import train_test_split

df["Amount_log"] = np.log1p(df["Amount"])
X = df.drop(columns=["Class", "Amount"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [3]:
#Treinar
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

modelos = {
    "logreg": LogisticRegression(max_iter=1000),
    "rf": RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=42),
    "xgb": XGBClassifier(scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
                         eval_metric="aucpr", random_state=42),
}

In [5]:
# Avaliar
from sklearn.metrics import classification_report, PrecisionRecallDisplay, RocCurveDisplay
from sklearn.preprocessing import StandardScaler

# Padronizar os dados
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

for nome, m in modelos.items():
    m.fit(X_train_scaled, y_train)   # use os dados já padronizados
    print(nome)
    print(classification_report(y_test, m.predict(X_test_scaled), digits=3))

logreg
              precision    recall  f1-score   support

           0      0.999     1.000     1.000     56864
           1      0.831     0.653     0.731        98

    accuracy                          0.999     56962
   macro avg      0.915     0.826     0.866     56962
weighted avg      0.999     0.999     0.999     56962

rf
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     56864
           1      0.961     0.745     0.839        98

    accuracy                          1.000     56962
   macro avg      0.980     0.872     0.919     56962
weighted avg      0.999     1.000     0.999     56962

xgb
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     56864
           1      0.871     0.827     0.848        98

    accuracy                          0.999     56962
   macro avg      0.935     0.913     0.924     56962
weighted avg      0.999     0.999     0.999     56962



In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([("scaler", StandardScaler()),
                 ("clf", RandomForestClassifier(class_weight="balanced", random_state=42))])

grade = {"clf__n_estimators": [100, 200], "clf__max_depth": [None, 10]}
busca = GridSearchCV(pipe, grade, scoring="recall", cv=3, n_jobs=-1)
busca.fit(X_train, y_train)

proba = busca.predict_proba(X_test)[:, 1]
pred_03 = (proba >= 0.3).astype(int)   # compare com 0.5

In [8]:
import shap
explainer = shap.TreeExplainer(busca.best_estimator_.named_steps["clf"])
amostra = busca.best_estimator_.named_steps["scaler"].transform(X_test.iloc[:500])
shap_values = explainer.shap_values(amostra)